# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

Данные: https://disk.yandex.ru/d/6d5hFHvpAZjQdw

Ваша задача -- предсказать колонку relevance, используя все остальные данные об организации. Загрузим данные и посмотрим на них

In [ ]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [ ]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [ ]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [ ]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [ ]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [ ]:
eval_data.to_excel("eval_data.xlsx")

## Импорты

In [ ]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool

import json
import re
from typing import TypedDict, Dict, Any, List, Optional, Literal
import os
import time, uuid
import math
from dotenv import load_dotenv

In [ ]:
load_dotenv()

True

# Модульный агент: main и fackchecker

In [ ]:
train_data.iloc[3022]

,3592
Text,итальянский ресторан в москве рейтинг
address,"Москва, Ломоносовский проспект, 9/75"
name,Lomonosov; Ломоносов
normalized_main_rubric_name_ru,Ресторан
permalink,46653755361
prices_summarized,Ресторан Lomonosov предлагает широкий выбор бл...
relevance,1.0
reviews_summarized,"Организация занимается ресторанным бизнесом, п..."


In [ ]:
Q = ''

TavilySearchResults(
    max_results=3,
    search_depth="basic",
    include_answer=False,
    include_raw_content=False,
    include_images=False,
).invoke({"query": Q})

[{'title': 'Заказать Ушастик (Детское меню), 520 ... - Dostavka-eda',
  'url': 'https://www.dostavka-eda.com/moscow/place/chacha_mnyovniki_21/menu/3008924427_ushastik/',
  'content': 'Доставка "Ушастик" на дом из ChaCha в Москве по адресу улица Мнёвники, 21. Ушастик. Жареные котлеты из курицы (куриное бедро, лук репчатый, соль, перец)',
  'score': 0.9999887},
 {'title': 'Заказать Чурчхела с грецким орехом (Десерты ... - Dostavka-eda',
  'url': 'https://dostavka-eda.com/moscow/place/chacha_mnyovniki_21/menu/5000000014075692_churchkhela-s-gretskim-orekhom/',
  'content': 'Доставка "Чурчхела с грецким орехом" на дом из ChaCha в Москве по адресу улица Мнёвники, 21. Чурчхела с грецким орехом. Сладость из сгущенного фруктового сока с',
  'score': 0.9999826},
 {'title': 'Заказать Салат с тунцом и томатами (Летнее ... - Dostavka-eda',
  'url': 'https://www.dostavka-eda.com/moscow/place/chacha_mnyovniki_21/menu/3021596789_salat-s-tuntsom-i-tomatami/',
  'content': 'Доставка "Салат с тунцом и то

In [ ]:
# Идея: добавить гуглер для повышения знаний main модели
# Для того чтобы формулировать верные факты
# Агаповка это населенные пункт или название магазина?


In [ ]:
import os, re, json, math
from typing import Any, Dict, List, Optional, TypedDict, Literal, Set

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END


# -----------------------------
# Config
# -----------------------------
arcee_trinity_model = "arcee-ai/trinity-large-preview:free"

stepfun_model = 'stepfun/step-3.5-flash:free'
MODEL_NAME_MAIN = arcee_trinity_model
MODEL_NAME_FACT = arcee_trinity_model
ALLOWED_FACT_TYPES = {"objective", "constraint", "subjective"}

TEMPERATURE_MAIN = 0.5
TEMPERATURE_FACT = 0.5

TIMEOUT_S = 90
MAX_RETRIES = 2

MAX_FACTCHECK_CALLS = 2
FACTCHECK_MAX_SEARCH_CALLS = 3   # внутри фактчеккера (сколько он может сделать поисков)


# -----------------------------
# Helpers
# -----------------------------
def _clean_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and math.isnan(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none"} else s

def make_card_text(row: Dict[str, Any], max_reviews_len: int = 5000) -> str:
    """Собираем card_text из полей карточки (без label/relevance)."""
    name = _clean_str(row.get("name", ""))
    address = _clean_str(row.get("address", ""))
    rubric = _clean_str(row.get("normalized_main_rubric_name_ru", row.get("rubric", "")))
    permalink = _clean_str(row.get("permalink", ""))
    prices = _clean_str(row.get("prices_summarized", ""))
    reviews = _clean_str(row.get("reviews_summarized", ""))

    lines = ["Информация об организации:"]
    if name: lines.append(f"- Названия (через ;): {name}")
    if rubric: lines.append(f"- Рубрика: {rubric}")
    if address: lines.append(f"- Адрес: {address}")
    if permalink: lines.append(f"- Permalink: {permalink}")
    if prices: lines.append(f"- Услуги/цены (summary): {prices}")
    if reviews: lines.append(f"- Отзывы (summary): {reviews[:max_reviews_len]}")
    return "\n".join(lines).strip()

def extract_reviews_from_card(card_text: str) -> str:
    """Достаём блок отзывов из card_text (как минимум summary)."""
    # Простой эвристический парсер под твой формат make_card_text
    m = re.search(r"- Отзывы \(summary\):\s*(.*)$", card_text, flags=re.S)
    return (m.group(1).strip() if m else "").strip()

def parse_int_label(text: str) -> Optional[int]:
    s = (text or "").strip()
    return int(s) if s in {"0", "1"} else None

def parse_json_obj(text: str) -> Optional[dict]:
    if not text:
        return None
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.I)
        s = re.sub(r"\s*```$", "", s)
        s = s.strip()
    if not (s.startswith("{") and s.endswith("}")):
        return None
    try:
        return json.loads(s)
    except Exception:
        return None

def parse_json_list(text: str) -> Optional[list]:
    if not text:
        return None
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.I)
        s = re.sub(r"\s*```$", "", s)
        s = s.strip()
    if not (s.startswith("[") and s.endswith("]")):
        return None
    try:
        return json.loads(s)
    except Exception:
        return None

import time
from openai import InternalServerError

def safe_llm_invoke(llm: ChatOpenAI, messages: List[Any], *, max_attempts: int = 3) -> AIMessage:
    """Безопасно вызывает LLM. Повторяет попытку при перегрузе провайдера (503)."""
    last_error: Exception | None = None

    for attempt in range(1, max_attempts + 1):
        try:
            return llm.invoke(messages)
        except InternalServerError as e:
            last_error = e
            # Пытаемся достать retry_after_seconds из текста ошибки (в лоб)
            retry_seconds = 2
            try:
                msg = str(e)
                m = re.search(r"retry_after_seconds['\"]?:\s*(\d+)", msg)
                if m:
                    retry_seconds = int(m.group(1))
            except Exception:
                pass

            if attempt == max_attempts:
                raise
            time.sleep(retry_seconds)
        except Exception as e:
            # Любая другая ошибка — сразу пробрасываем
            raise

    # До сюда не дойдём
    raise last_error if last_error else RuntimeError("LLM invoke failed")


def tavily_results_sorting_func(result: dict, query: str) -> tuple:
    """Выдает сортировочный скор одному результату веб-поиска"""
    query_words = query.lower().split()
    sorting_score = [0] + [float(result.get("score", 0))]
    for word in query_words:
        if word in result.get("title", "").lower():
            sorting_score[0] += 3
        if word in result.get("content", "").lower():
            sorting_score[0] += 2
    return tuple(sorting_score)

def preprocess_tavily_results(raw: dict, used_urls_total: set, query: str, max_results: int = 5, max_content_len: int = 1000,) -> str | Dict[str, str|set]:
    """ Предобрабатывает результаты Tavily """

    if not isinstance(raw, dict):
        return "Поиск не вернул результатов."

    results = raw.get("results") or []
    if not results:
        return "Поиск не дал результатов."

    cleaned = []
    used_urls = set()

    results.sort(key = lambda x: tavily_results_sorting_func(x, query), reverse = True)

    for r in results:
        title = r.get("title", "")
        url = r.get("url", "")
        if url in used_urls_total:
            continue
        used_urls.add(url)
        content = (r.get("content") or "")[:max_content_len]
        cleaned.append(
            f"Заголовок: {title}\n"
            f"Текст: {content}\n"
        )
        if len(cleaned) >= max_results:
            break

    return {'result' : 'Результат веб-поиска:\n ' + "\nNEXT SOURCE\n".join(cleaned),
            'used_urls' : used_urls_total.union(used_urls)}


# -----------------------------
# Tools (Tavily)
# -----------------------------
tavily = TavilySearchResults(
    max_results=10,
    search_depth="basic",
    include_answer=False,
    include_raw_content=False,
    include_images=False,
    tavily_api_key=os.environ.get("TAVILY_API_KEY", ""),
)


# -----------------------------
# LLMs
# -----------------------------
llm_main = ChatOpenAI(
    model=MODEL_NAME_MAIN,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=TEMPERATURE_MAIN,
    timeout=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    max_tokens=4000,
)

llm_fact = ChatOpenAI(
    model=MODEL_NAME_FACT,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=TEMPERATURE_FACT,
    timeout=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    max_tokens=4000,
).bind_tools([tavily])


# -----------------------------
# Prompts
# -----------------------------
MAIN_SYSTEM = """
Ты — агент бинарной классификации релевантности организации рубричному запросу.

Вход:
- query: запрос пользователя
- card_text: карточка организации
- org_name, org_address: точное имя и полный адрес (с городом)

Твоя логика:

1) Сформулируй список фактов, которые должны быть выполнены, чтобы организация была релевантна запросу, исходя из query.
   То есть тебе нужно собрать качества, которые ищет пользователем своим запросом query
   fact_text Должен быть четким, понятным и нетривиальным,
   то есть не у каждого объекта этой рубрики должен быть этот факт.
   Обычно запрос формирует 1–2 факта, редко 3-4.

   Каждый факт с полями:
   - type: "objective" | "constraint" | "subjective"
   - fact_text: кратко и проверяемо

   Как выбирать поля:
   type:
   - objective : для объективных фактов (например определенная услуга/товар/режим работы/адрес итд).
   - constraint : Четкое числовое органичение (например возраст до 7 лет, Средний чек до 3000 рублей, Расстояние до метро до 1 км итд).
   - subjective : Субъективные характеристики - мнение или оценка людей (например хороший, дешево, романтично, вкусно итд).

   Примеры фактов по query:
   1. query : физкультурный колледж в москве после 9 класса
    "facts" : [{"type": "objective","fact_text": "Организация является колледжем"},
               {"type": "objective","fact_text": "Организация расположена в Москве"},
               {"type": "objective","fact_text": "Колледж принимает выпускников 9 класса (со средним основным образованием)"}]
  2. query : танцы для детей от 1.5 года
     "facts" : [{"type": "objective","fact_text": "Организация оказывает услуги по обучению детей танцам"},
                {"type": "constraint","fact_text": "Организация учит танцам детей от 1.5 лет"}]
  3. query : где покушать недорого
     "facts" : [{"type": "objective","fact_text": "Организация продает еду"},
                {"type": "subjective","fact_text": "Организация считаетсяя дешевой"}]
  4. query : снять квартиру в приморском крым цены 2018
     "facts" : [{"type": "objective","fact_text": "Организация находится в населенном пункте Приморский, в Крыму"},
                {"type": "objective","fact_text": "Организация предоставляет услуги съема квартиры"}]
  5. query : 628695 почтовое отделение
     "facts" : [{"type": "constraint","fact_text": "Организация является почтовым отделением с индексом 628695"}]
  6. query : грузинский ресторан мневники 21
     "facts" : [{"type": "objective","fact_text":"Организация является рестораном грузинской кухни"},
                {"type": "objective","fact_text":"Адрес организации — мневники, дом 21"}]
  7. query : автомобильный завод выпускает 16326 автомобилей в год
     "facts" : [{"type": "objective","fact_text":"Организация является автомобильным заводом"},
                {"type": "constraint","fact_text":"Организация выпускает 16326 автомобилей в год"}]
  8. query : аукцион продажи машин со штрафстоянок в спб
    "facts" : [{"type": "objective","fact_text": "Организация расположена в Санкт-Петербурге"},
               {"type": "objective","fact_text": "Организация проводит аукционы по продаже машин со штрафстоянок"}]
  9. query : кафе центр нижневартовска недорого
    "facts" : [{"type": "objective","fact_text": "Организация является кафе или рестораном"},
               {"type": "objective","fact_text": "Организация находится центре города Нижневартовск"},
               {"type": "subjective","fact_text": "Организация считается недорогой (дешевой)"}]


2) Запрещено добавлять тривиальные факты, которые по умолчанию верны
для любой организации данной рубрики.
Примеры запрещённых фактов:
- "Аптека продаёт лекарственные препараты"
- "Ресторан готовит еду"
- "Колледж обучает студентов"
- "Магазин продаёт товары"
Факт должен быть именно тем, что отличает релевантную организацию
от нерелевантной по данному запросу.

3) Не воспринимай запрос слишком буквально.
Не добавляй в fact_text артефакты запроса, которые не являются фильтром выбора организации на Картах.

Примеры артефактов запроса (не должно влиять на формирование фактов):
  1. запрос = "итальянский ресторан в москве рейтинг",
     артефакт : "рейтинг"
  2. запрос = "купить разогревающую мазь цена за кг"
     артефакт : "цена за кг"
  3. запрос = "доставка из китая какие отзывы какая стоимость за кубометр",
     артефакт : "какие отзывы какая стоимость за кубометр"

Если запрос содержит такие детали, игнорируй их
и формулируй факт только по сути запроса.

3) Затем проверь каждый факт по card_text:
   - Если факт явно подтверждён — ок.
   - Если явно опровергнут — верни "0".
   - Если не хватает данных — ты можешь вызвать tool fact_check ДЛЯ ОДНОГО факта (иногда второго), максимум 2 вызова.
   - Если type=subjective: сначала ищи подтверждение в отзывах внутри card_text. Если отзывов недостаточно — можно отправить факт чеккеру (он получит все отзывы).

4) Финальный ответ:
   - Верни "1" только если все важные факты подтверждены.
   - Если остались неподтверждённые факты — верни "0".

Формат общения:
- Сначала (в первом сообщении) верни ТОЛЬКО JSON c фактами (без пояснений).
- После этого на следующих шагах: либо вызывай tool fact_check, либо верни финальный ответ строго "0" или "1".

Шаг 1 (СЕЙЧАС) JSON c фактами:
Верни ТОЛЬКО один JSON-объект строгого формата:
{
  "org_name": "<org_name из входа без изменений>",
  "org_address": "<org_address из входа без изменений>",
  "facts": [
    {"type": "objective|constraint|subjective", "fact_text": "..."}
  ]
}

"""

FACTCHECK_SYSTEM = f"""
Ты — факт-чеккер. Тебе дают факт и конкретную организацию (название, полный адрес).
Задача: проверить, подтверждается ли факт именно для этой КОНКРЕТНОЙ организации. (Именно с этим адресом и названием)
Разрешено использовать веб-поиск tavily_search_results_json.

Тебе дают:
- fact_text: один конкретный факт
- type: тип факта (objective | constraint | subjective)
- org_name: название организации или эквивалентные названия через символ ";"
- org_address: полный адрес организации с городом
- reviews (если type=subjective)

Правила:
- Если результат веб-поиска никак не связан с организацией, то его нельзя использовать
- Максимум {FACTCHECK_MAX_SEARCH_CALLS} веб-запроса.
- В запросе нужно использовать: org_name или org_address (или город/улица из адреса) и ключевые слова из fact_text.
- Подтверждать факт можно только если найденная информация относится к этой организации:
  совпадает название и/или адрес.
- Если type=subjective и в reviews есть явные подтверждения факта — можешь вернуть true без веб-поиска.
- Если подтверждения нет — ответ всегда false.
- Выход: строго "true": (когда факт подтвердился) или "false": (когда факт опровергся или не удалось подтвердить).

ВАЖНО:
Если первый веб-поиск не дал результатов (не поддтвердил и не опрверг fact_text), то нужно сделать следующий.
Нужно ЗНАЧИТЕЛЬНО поменять

Если второй веб-поиск не дал результатов (не поддтвердил и не опрверг fact_text),
то нужно сделать финальный веб-поиск в back step запросом.
back step запрос - это более общий вопрос.
Пример back step запросов:
обычный запрос:


ВАЖНО: Твои действя сейчас:
- Либо вызвать tavily_search_results_json
- Либо вернуть СТОРО ТОЛЬКО "true" или "false" и больше ничего!
"""


# -----------------------------
# State
# -----------------------------
FactType = Literal["objective", "constraint", "subjective"]

class Fact(TypedDict):
    type: FactType
    fact_text: str

class AgentState(TypedDict, total=False):
    query: str
    card_text: str
    org_name: str
    org_address: str
    reviews: str

    # main loop
    facts: List[Fact]
    facts_checked: List[Dict[str, Any]]   # [{"fact": Fact, "supported": bool, "source": "card|factcheck"}]
    next_fact_idx: int

    factcheck_calls: int
    pending_fact: Optional[Fact]
    pending_fact_idx: Optional[int]
    pending_fact_supported: Optional[bool]

    final_label: Optional[int]
    stop_reason: Optional[str]

    messages_main: List[Any]
    messages_fact: List[Any]
    used_urls: Set[str]


# -----------------------------
# FactCheck tool wrapper (node-level)
# -----------------------------
from typing import Any

def run_factchecker(
    fact: Fact,
    org_name: str,
    org_address: str,
    reviews: str,
    used_urls_total: set[str],
) -> tuple[bool, list[Any], set[str]]:
    """Вызывает факт-чеккер LLM. Для subjective дополнительно передаёт reviews."""
    payload = {
        "type": fact["type"],
        "fact_text": fact["fact_text"],
        "org_name": org_name,
        "org_address": org_address,
    }
    if fact["type"] == "subjective":
        payload["reviews"] = reviews or ""

    messages: list[Any] = [
        SystemMessage(content=FACTCHECK_SYSTEM),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False))
    ]

    search_calls = 0
    max_turns = 12  # защита от вечного цикла
    turns = 0

    # локальная копия set, чтобы не мутировать внешний объект неожиданно
    used_urls = set(used_urls_total)

    while True:
        turns += 1
        if turns > max_turns:
            return False, messages, used_urls

        assistant_message = safe_llm_invoke(llm_fact, messages)
        messages.append(assistant_message)

        answer = (getattr(assistant_message, "content", "") or "").strip().lower()

        # строго true/false (лучше так, чем "true" in answer)
        if answer in {"true", "false"}:
            return (answer == "true"), messages, used_urls

        tool_calls = getattr(assistant_message, "tool_calls", None) or []
        if not tool_calls:
            messages.append(
                HumanMessage(
                    content=(
                        "Формат нарушен.\n"
                        "Сделай ОДНО из двух:\n"
                        "1) Верни строго true или false\n"
                        "2) Либо вызови tavily_search_results_json с {\"query\": \"...\"}\n"
                        "Без пояснений."
                    )
                )
            )
            continue

        executed_any = False
        for call in tool_calls:
            if call.get("name") != "tavily_search_results_json":
                continue

            if search_calls >= FACTCHECK_MAX_SEARCH_CALLS:
                messages.append(HumanMessage(content="Лимит поиска исчерпан. Верни строго true или false."))
                executed_any = True
                continue

            query = str((call.get("args") or {}).get("query", "")).strip()
            raw = tavily.invoke({"query": query})

            pre = preprocess_tavily_results(
                raw=raw,
                used_urls_total=used_urls,
                query=query,
                max_results=5,
                max_content_len=1000,
            )

            tool_text = pre["result_text"]
            used_urls = pre["used_urls"]

            messages.append(ToolMessage(content=tool_text, tool_call_id=call.get("id")))
            search_calls += 1
            executed_any = True

        if not executed_any:
            return False, messages, used_urls



# -----------------------------
# Nodes
# -----------------------------
def build_initial_state(row: Dict[str, Any]) -> AgentState:
    query = _clean_str(row.get("Text", ""))
    card_text = make_card_text(row, max_reviews_len=5000)
    org_name = _clean_str(row.get("name", ""))
    org_address = _clean_str(row.get("address", ""))
    reviews = extract_reviews_from_card(card_text)

    return AgentState(
        query=query,
        card_text=card_text,
        org_name=org_name,
        org_address=org_address,
        reviews=reviews,
        facts=[],
        facts_checked=[],
        next_fact_idx=0,
        factcheck_calls=0,
        pending_fact=None,
        pending_fact_idx=None,
        pending_fact_supported=None,
        final_label=None,
        stop_reason=None,
        messages_main=[
            SystemMessage(content=MAIN_SYSTEM),
            HumanMessage(content=f"query: {query}\norg_name: {org_name}\norg_address: {org_address}\n\ncard_text:\n{card_text}")
        ],
        messages_fact=[],
        used_urls = set(),
    )

def main_generate_facts_node(state: AgentState) -> AgentState:
    """Первый шаг: главный агент возвращает JSON-массив фактов."""
    resp = safe_llm_invoke(llm_main, state["messages_main"]) # resp = llm_main.invoke(state["messages_main"])
    state["messages_main"].append(resp)

    obj = parse_json_obj(getattr(resp, "content", None))
    if not isinstance(obj, dict):
        state["final_label"] = 0
        state["stop_reason"] = "facts_parse_failed"
        return state

    # строго сверяем org_name/org_address (или просто берем из state)
    facts_raw = obj.get("facts")
    if not isinstance(facts_raw, list):
        print('Main не вернул list')
        state["final_label"] = 0
        state["stop_reason"] = "facts_no_list"
        return state

    facts: List[Fact] = []
    for fact in facts_raw:
        if not isinstance(fact, dict):
            continue
        fact_type = fact.get("type")
        fact_text = (fact.get("fact_text") or "").strip()
        if fact_type not in {"objective", "constraint", "subjective"} or not fact_text:
            continue
        facts.append(Fact(type=fact_type, fact_text=fact_text))

    if not facts:
        state["final_label"] = 0
        state["stop_reason"] = "no_facts"
        return state

    state["facts"] = facts
    state["stop_reason"] = "facts_ok"
    return state

def _parse_factcheck_call(obj: dict) -> Optional[Dict[str, str]]:
    """ Парсит вызов Факт-чеккера """
    call = obj.get("call_fact_check")
    if not isinstance(call, dict):
        return None
    fact_type = call.get("type")
    fact_text = (call.get("fact_text") or "").strip()
    if fact_type not in ALLOWED_FACT_TYPES:
        return None
    if not fact_text:
        return None
    return {"type": fact_type, "fact_text": fact_text}

def main_evaluate_card_node(state: AgentState) -> AgentState:
    """
    Главный агент решает:
    - что подтверждено карточкой
    - что требует фактчека
    - или финальный ответ
    """
    if state.get("final_label") in (0, 1):
        return state

    context_payload = {
        "org_name": state["org_name"],
        "org_address": state["org_address"],
        "facts": state.get("facts", []),
        "already_checked": state.get("facts_checked", []),
        "factcheck_calls": int(state.get("factcheck_calls", 0)),
        "max_factcheck_calls": MAX_FACTCHECK_CALLS,
    }

    # Explicitly demand machine format here.
    state["messages_main"].append(
        HumanMessage(
            content=(
                "Оцени факты по card_text и already_checked.\n\n"
                "Верни ОДНО из двух (строго):\n"
                "1) финальный ответ: 0 или 1\n"
                "2) JSON для проверки одного факта:\n"
                "{\"call_fact_check\": {\"type\": \"objective|constraint|subjective\", \"fact_text\": \"...\"}}\n\n"
                "Никаких пояснений.\n\n"
                f"КОНТЕКСТ:\n{json.dumps(context_payload, ensure_ascii=False)}"
            )
        )
    )

    def run_and_parse_once() -> Dict[str, Any]:
        assistant_message = safe_llm_invoke(llm_main, state["messages_main"]) # assistant_message = llm_main.invoke(state["messages_main"])
        state["messages_main"].append(assistant_message)

        content = getattr(assistant_message, "content", None)

        label = parse_int_label(content)
        if label is not None:
            return {"kind": "final", "label": label}

        obj = parse_json_obj(content)
        if isinstance(obj, dict):
            call = _parse_factcheck_call(obj)
            if call:
                return {"kind": "factcheck", "call": call}

        return {"kind": "invalid"}

    first = run_and_parse_once()
    if first["kind"] == "final":
        state["final_label"] = first["label"]
        state["stop_reason"] = "main_final"
        return state

    if first["kind"] == "factcheck":
        if int(state.get("factcheck_calls", 0)) >= MAX_FACTCHECK_CALLS:
            state["final_label"] = 0
            state["stop_reason"] = "factcheck_limit_reached"
            return state

        state["pending_fact"] = Fact(type=first["call"]["type"], fact_text=first["call"]["fact_text"])
        state["pending_fact_idx"] = None  # опционально: можно попытаться найти индекс
        state["stop_reason"] = "call_factcheck"
        return state

    # ---- repair-turn ----
    state["messages_main"].append(
        HumanMessage(
            content=(
                "Формат нарушен.\n"
                "Верни ТОЛЬКО:\n"
                "- либо один символ 0 или 1\n"
                "- либо JSON: {\"call_fact_check\": {\"type\": \"objective|constraint|subjective\", \"fact_text\": \"...\"}}\n"
                "Без пояснений."
            )
        )
    )

    second = run_and_parse_once()
    if second["kind"] == "final":
        state["final_label"] = second["label"]
        state["stop_reason"] = "main_final_after_repair"
        return state

    if second["kind"] == "factcheck":
        if int(state.get("factcheck_calls", 0)) >= MAX_FACTCHECK_CALLS:
            state["final_label"] = 0
            state["stop_reason"] = "factcheck_limit_reached_after_repair"
            return state

        state["pending_fact"] = Fact(type=second["call"]["type"], fact_text=second["call"]["fact_text"])
        state["pending_fact_idx"] = None
        state["stop_reason"] = "call_factcheck_after_repair"
        return state

    # still invalid => safe default
    state["final_label"] = 0
    state["stop_reason"] = "main_protocol_violation_default_0"
    return state


def factcheck_node(state: AgentState) -> AgentState:
    """Запускаем факт-чеккер и сохраняем результат."""
    if state.get("final_label") in (0, 1):
        return state

    fact = state.get("pending_fact")
    if not fact:
        state["final_label"] = 0
        state["stop_reason"] = "no_pending_fact"
        return state

    supported, fact_messages = run_factchecker(
          fact=fact,
          org_name=state["org_name"],
          org_address=state["org_address"],
          reviews=state.get("reviews", ""),
    )


    state["pending_fact_supported"] = bool(supported)
    state["factcheck_calls"] = int(state.get("factcheck_calls", 0)) + 1
    state["stop_reason"] = "factcheck_done"
    state["messages_fact"] = fact_messages
    return state


def apply_factcheck_result_node(state: AgentState) -> AgentState:
    """Заносим результат фактчека в facts_checked и чистим pending."""
    if state.get("final_label") in (0, 1):
        return state

    fact = state.get("pending_fact")
    supported = state.get("pending_fact_supported")
    if fact is None or supported is None:
        state["final_label"] = 0
        state["stop_reason"] = "apply_missing_data"
        return state

    state["facts_checked"].append({
        "fact": fact,
        "supported": bool(supported),
        "source": "factcheck",
    })

    # очистка pending
    state["pending_fact"] = None
    state["pending_fact_idx"] = None
    state["pending_fact_supported"] = None
    state["stop_reason"] = "applied"
    return state


# -----------------------------
# Routing
# -----------------------------
def route_after_generate_facts(state: AgentState) -> str:
    if state.get("final_label") in (0, 1):
        return "end"
    return "evaluate"

def route_after_evaluate(state: AgentState) -> str:
    if state.get("final_label") in (0, 1):
        return "end"
    if state.get("pending_fact"):
        return "factcheck"
    return "end"

def route_after_factcheck(state: AgentState) -> str:
    if state.get("final_label") in (0, 1):
        return "end"
    return "apply"

def route_after_apply(state: AgentState) -> str:
    if state.get("final_label") in (0, 1):
        return "end"
    # после применения снова идём к main_evaluate_card_node, он решит: второй фактчек или финал
    return "evaluate"


# -----------------------------
# Graph
# -----------------------------
graph = StateGraph(AgentState)

graph.add_node("generate_facts", main_generate_facts_node)
graph.add_node("evaluate", main_evaluate_card_node)
graph.add_node("factcheck", factcheck_node)
graph.add_node("apply", apply_factcheck_result_node)

graph.set_entry_point("generate_facts")

graph.add_conditional_edges("generate_facts", route_after_generate_facts, {"evaluate": "evaluate", "end": END})
graph.add_conditional_edges("evaluate", route_after_evaluate, {"factcheck": "factcheck", "end": END})
graph.add_conditional_edges("factcheck", route_after_factcheck, {"apply": "apply", "end": END})
graph.add_conditional_edges("apply", route_after_apply, {"evaluate": "evaluate", "end": END})

app = graph.compile()


from functools import lru_cache

def predict(row: Dict[str, Any] | pd.Series, debug: bool = False):
    state = build_initial_state(row)
    out = app.invoke(state)

    label = out.get("final_label")
    if label not in (0, 1):
        label = 0

    if debug:
        return int(label), out
    return int(label)


In [ ]:
label, output = predict(train_data.loc[1123], debug = True)

In [ ]:
output.keys()

dict_keys(['query', 'card_text', 'org_name', 'org_address', 'reviews', 'facts', 'facts_checked', 'next_fact_idx', 'factcheck_calls', 'pending_fact', 'pending_fact_idx', 'pending_fact_supported', 'final_label', 'stop_reason', 'messages_main', 'messages_fact'])

In [ ]:
output['messages_fact'][-2].content

'[{"title": "Максавит, Мончегорская, 15Ак1 в Нижнем Новгороде", "url": "https://nizhniy-novgorod.orgsinfo.ru/company/2614494-maksavit", "content": "# Максавит. ## аптека. #### О компании. **Максавит, аптека**: Нижний Новгород, Мончегорская, 15Ак1 (Автозаводский район, микрорайон Мончегорский). Подробную информацию можно узнать по телефону, на сайте. Для уточнения местоположения, где находится, как найти и проехать, - используйте карту и вид улицы. В справочнике Нижнего Новгорода компания размещена в рубрике медицина. Напишите отзыв или оставьте рекомендацию. Добавить подробное описание, фото, новости, объявления, изменить данные, установить точные координаты, доступно в личном кабинете после регистрации. #### Виды деятельности компании. #### Похожие компании рядом. #### Ближайший транспорт. 32   65   67   68   77   т44   т75   э22. #### Максавит. Все компании в Нижнем Новгороде по адресу: улица Мончегорская, 15А, корп.1. Рекомендуем уточнять актуальную информацию на сайте или по номеру

In [ ]:
print(output['messages_fact'][-2].content)

[{"title": "Максавит, Мончегорская, 15Ак1 в Нижнем Новгороде", "url": "https://nizhniy-novgorod.orgsinfo.ru/company/2614494-maksavit", "content": "# Максавит. ## аптека. #### О компании. **Максавит, аптека**: Нижний Новгород, Мончегорская, 15Ак1 (Автозаводский район, микрорайон Мончегорский). Подробную информацию можно узнать по телефону, на сайте. Для уточнения местоположения, где находится, как найти и проехать, - используйте карту и вид улицы. В справочнике Нижнего Новгорода компания размещена в рубрике медицина. Напишите отзыв или оставьте рекомендацию. Добавить подробное описание, фото, новости, объявления, изменить данные, установить точные координаты, доступно в личном кабинете после регистрации. #### Виды деятельности компании. #### Похожие компании рядом. #### Ближайший транспорт. 32   65   67   68   77   т44   т75   э22. #### Максавит. Все компании в Нижнем Новгороде по адресу: улица Мончегорская, 15А, корп.1. Рекомендуем уточнять актуальную информацию на сайте или по номеру 

In [ ]:
train_data.loc[1123]

,1123
Text,эспумизан в эмульсия цена за кг
address,"Нижний Новгород, Автозаводский район, микрорай..."
name,Максавит; Maksavit; УК Максавит
normalized_main_rubric_name_ru,Аптека
permalink,76965995350
prices_summarized,None
relevance,1.0
reviews_summarized,Организация занимается продажей лекарств и мед...
